In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install python-docx
from docx import Document
import openpyxl
from openpyxl.styles import Font, Alignment
import re
import os

# =============================================
# HELPER FUNCTIONS
# =============================================

bengali_digits = {'০':'0','১':'1','২':'2','৩':'3','৪':'4',
                  '৫':'5','৬':'6','৭':'7','৮':'8','৯':'9'}

def bn_to_int(bn_str):
    return int(''.join(bengali_digits.get(c, c) for c in bn_str))

def clean_text(text):
    """Remove leading/trailing whitespace and normalize internal newlines"""
    text = text.strip()
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

def is_bold_para(para):
    runs = [r for r in para.runs if r.text.strip()]
    if not runs:
        return False
    return all(r.bold for r in runs)

def save_xlsx(filename, headers, rows):
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Sheet1"

    # Header style
    header_font = Font(name='Arial', bold=True, size=11)
    cell_font = Font(name='Arial', size=11)
    wrap_align = Alignment(wrap_text=True, vertical='top')

    # Write headers
    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.font = header_font
        cell.alignment = wrap_align

    # Write data rows
    for row_idx, row_data in enumerate(rows, 2):
        for col_idx, value in enumerate(row_data, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.font = cell_font
            cell.alignment = wrap_align

    # Auto-width columns
    ws.column_dimensions['A'].width = 8
    ws.column_dimensions['B'].width = 80

    output_path = f"/content/drive/MyDrive/JOb Work/Python Code/{filename}"
    wb.save(output_path)
    print(f"✅ Saved: {output_path} ({len(rows)} rows)")
    return output_path


# =============================================
# LOAD DOCUMENT
# =============================================

doc = Document('/content/drive/MyDrive/JOb Work/Python Code/Book.docx')

# Build paragraph list with metadata
paras = []
for i, para in enumerate(doc.paragraphs):
    text = para.text.strip()
    bold = is_bold_para(para)
    paras.append({'idx': i, 'text': text, 'bold': bold})


# =============================================
# FILE 1: অধ্যায় (Chapters)
# =============================================

chapter_pattern = re.compile(r'^অধ্যায়[:\s]')
chapters = []
chapter_id = 1

for p in paras:
    if not p['text']:
        continue
    if p['bold'] and chapter_pattern.match(p['text']):
        name = clean_text(p['text'])
        chapters.append([chapter_id, name])
        chapter_id += 1

save_xlsx("chapters.xlsx", ["id", "name"], chapters)
print(f"   Chapters found: {[r[1][:40] for r in chapters]}\n")


# =============================================
# FILE 2: Sub-sections
# =============================================

bold_indices = [i for i, p in enumerate(paras) if p['bold'] and p['text'] and not chapter_pattern.match(p['text'])]

subsections = []
sub_id = 1

for p in paras:
    if not p['text']:
        continue
    # Bold, not a chapter
    if p['bold'] and not chapter_pattern.match(p['text']):
        name = clean_text(p['text'])
        subsections.append([sub_id, name])
        sub_id += 1

save_xlsx("subsections.xlsx", ["id", "name"], subsections)
print(f"   Subsections found: {[r[1][:40] for r in subsections]}\n")


# =============================================
# FILE 3: Hadiths
# =============================================

hadith_start_pattern = re.compile(r'^\[([০-৯]+)\]')

hadith_starts = []
for i, p in enumerate(paras):
    if not p['text']:
        continue
    m = hadith_start_pattern.match(p['text'])
    if m and not p['bold']:
        hadith_num = bn_to_int(m.group(1))
        hadith_starts.append((i, hadith_num))

# Second pass: for each hadith, collect text until next hadith start OR bold paragraph (sub-section/chapter)
hadiths = []

for k, (start_idx, hadith_num) in enumerate(hadith_starts):
    # Determine end boundary
    if k + 1 < len(hadith_starts):
        next_hadith_idx = hadith_starts[k + 1][0]
    else:
        next_hadith_idx = len(paras)

    # Collect text parts for this hadith
    text_parts = []
    for j in range(start_idx, next_hadith_idx):
        p = paras[j]
        if not p['text']:
            continue
        # Stop if we hit a bold paragraph (sub-section or chapter) — but skip the starting para's own check
        if j > start_idx and p['bold']:
            break
        text_parts.append(p['text'])

    # Join all parts into one string, remove newlines/blanks
    full_text = ' '.join(text_parts)
    full_text = clean_text(full_text)

    # Remove the leading [number] marker from the hadith text
    full_text = hadith_start_pattern.sub('', full_text).strip()

    hadiths.append([hadith_num, full_text])

# Sort by hadith number
hadiths.sort(key=lambda x: x[0])

save_xlsx("hadiths.xlsx", ["id", "hadith"], hadiths)
print(f"   Hadiths found: {len(hadiths)} entries")
for h in hadiths:
    print(f"   [{h[0]}] {h[1][:60]}...")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 22.5 MB/s eta 0:00:00
✅ Saved: /content/drive/MyDrive/JOb Work/Python Code/chapters.xlsx (1 rows)
   Chapters found: ['অধ্যায়: পিতা-মাতার সাথে সদ্ব্যবহার']

✅ Saved: /content/drive/MyDrive/JOb Work/Python Code/subsections.xlsx (15 rows)
   Subsections found: ['আমি মানুষকে তার পিতা-মাতার সাথে সদ্ব্যবহ', 'মায়ের সাথে সদাচরণ', 'বাবার সাথে সদাচরণ', 'পিতা-মাতা অত্যাচার করলেও তাদের সাথে সদাচ', 'মাতা-পিতার সাথে নরম সুরে কথা বলা', 'পিতা-মাতার অবাধ্য হওয়ার পরিণতি', 'পাপ ব্যতীত পিতা-মাতার সব বিষয়ে আনুগত্য ', 'যে ব্যক্তি পিতা-মাতাকে পেল কিন্তু জান্না', 'যে তার পিতা-মাতার সাথে ভালো আচরণ করবে আল', 'অমুসলিম পিতার সাথেও সদাচরণ করা আবশ্যক', 'পিতা-মাতার অবাধ্য হওয়ার শাস্তি', 'পিতা-মাতার ক্রন্দন', 'মাতা-পিতার দুআ', 'খ্রিষ্টান মা-কে ইসলামের দাওয়াত দেয়া', 'পিতা-মাতাকে গালি না দেয়া']

✅ Saved: /content/drive/MyDrive/JOb Work/Python Code/hadiths.xlsx (33 rows)
   Hadiths found: 33 entries
   [1] আমর ইবনু শাইবানি রাহিমাহুল্লাহু বলেন—আমাদের কাছ